# 00 - Data Inventory & Environment Verification

**Title**: `00_data_inventory.ipynb`  
**Purpose**: Verify computing environment, installed package versions, hardware/disk capacity, raw dataset integrity, Socrata 311 schemas, and processed cube date coverage.  
**Inputs**: `data/raw/311_Service_Requests_from_2020_to_Present_20261107.csv`, `data/processed/311_cube.csv`  
**Outputs**: Environment specification, raw file inventory, sample row inspection, binary tail verification, API field presence table  
**Runtime**: ~5-10 seconds

In [1]:
import sys; from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists()); sys.path.insert(0, str(ROOT))
from src.nbutils import Checker, md
RUN_HEAVY = False
RUN_API = False

In [2]:
from src.inventory import get_environment_info
import pandas as pd

env_info = get_environment_info(ROOT)
env_df = pd.DataFrame([
    {"Component": "Python Version", "Value": env_info["python"]},
    {"Component": "pandas", "Value": env_info["pandas"]},
    {"Component": "numpy", "Value": env_info["numpy"]},
    {"Component": "scikit-learn", "Value": env_info["scikit-learn"]},
    {"Component": "scipy", "Value": env_info["scipy"]},
    {"Component": "librosa", "Value": env_info["librosa"]},
    {"Component": "torch", "Value": env_info["torch"]},
    {"Component": "fastapi", "Value": env_info["fastapi"]},
    {"Component": "osmnx", "Value": env_info["osmnx"]},
    {"Component": "ffmpeg", "Value": env_info["ffmpeg"]},
    {"Component": "Git Branch", "Value": env_info["git_branch"]},
    {"Component": "Free Disk (GB on C:)", "Value": str(env_info["free_disk_gb"])},
])
print("=== Computing Environment & Software Stack ===")
print(env_df.to_string(index=False))

=== Computing Environment & Software Stack ===
           Component                                                                                                          Value
      Python Version                                                                                                        3.11.18
              pandas                                                                                                          3.1.1
               numpy                                                                                                          2.4.6
        scikit-learn                                                                                                          1.9.1
               scipy                                                                                                         1.17.1
             librosa                                                                                                         0.11.0
               torch         

In [3]:
from src.inventory import get_raw_inventory

raw_inv = get_raw_inventory(ROOT / "data" / "raw")
raw_inv_df = pd.DataFrame(raw_inv)
print("=== data/raw File Inventory ===")
print(raw_inv_df.to_string(index=False))

=== data/raw File Inventory ===
                                              filename  size_gb            modified
                                              .gitkeep     0.00 2026-11-07 10:40:54
311_Service_Requests_from_2020_to_Present_20261107.csv     3.47 2026-11-06 21:09:50


In [4]:
from src.inventory import inspect_raw_csv_sample

raw_csv_path = ROOT / "data" / "raw" / "311_Service_Requests_from_2020_to_Present_20261107.csv"
raw_inspection = inspect_raw_csv_sample(raw_csv_path)

print(f"Header field count: {raw_inspection['header_count']}")
print(f"Date-parse failures on sample (format '%m/%d/%Y %I:%M:%S %p'): {raw_inspection['date_parse_failures']}")

print("\nConcept to Real Column Mappings:")
mapping_df = pd.DataFrame([
    {"Concept": k, "Real Column": v, "Present in Header": raw_inspection["mapping_status"][k]}
    for k, v in raw_inspection["concept_mappings"].items()
])
print(mapping_df.to_string(index=False))

print("\nFirst 5 Sample Rows (selected columns):")
sample_cols = ["Created Date", "Borough", "Problem (formerly Complaint Type)", "Problem Detail (formerly Descriptor)"]
avail_cols = [c for c in sample_cols if c in raw_inspection["sample_rows"].columns]
print(raw_inspection["sample_rows"][avail_cols].head(5).to_string(index=False))

print("\nLast 3 lines in binary mode:")
for i, line in enumerate(raw_inspection["last_3_lines"], 1):
    print(f"  [{i}] {repr(line)}")

print(f"\nCompletion verdict: {raw_inspection['verdict']}")

Header field count: 44
Date-parse failures on sample (format '%m/%d/%Y %I:%M:%S %p'): 0

Concept to Real Column Mappings:
               Concept                          Real Column  Present in Header
          created date                         Created Date               True
           closed date                          Closed Date               True
        complaint type    Problem (formerly Complaint Type)               True
            descriptor Problem Detail (formerly Descriptor)               True
                status                               Status               True
resolution description               Resolution Description               True
               borough                              Borough               True
              latitude                             Latitude               True
             longitude                            Longitude               True
      incident address                     Incident Address               True
         

In [5]:
from src.inventory import get_cube_coverage

cube_path = ROOT / "data" / "processed" / "311_cube.csv"
cube_info = get_cube_coverage(cube_path)
print("=== Date Coverage from 311_cube.csv ===")
print(f"First year_month : {cube_info['first_month']} (partial month: {cube_info['first_month'] in cube_info['partial_months']})")
print(f"Last year_month  : {cube_info['last_month']} (partial month: {cube_info['last_month'] in cube_info['partial_months']})")
print(f"All valid months : {', '.join(cube_info['all_months'])}")

=== Date Coverage from 311_cube.csv ===
First year_month : 2025-10 (partial month: True)
Last year_month  : 2026-11 (partial month: True)
All valid months : 2025-10, 2025-11, 2025-12, 2026-01, 2026-02, 2026-03, 2026-04, 2026-05, 2026-06, 2026-07, 2026-08, 2026-09, 2026-10, 2026-11


In [6]:
from src.inventory import check_socrata_fields

api_results = check_socrata_fields(["76ig-c548", "erm2-nwe9"])
all_unreachable = all(info["status_code"] != 200 for info in api_results.values())
if all_unreachable:
    print("NYC 311 Socrata API is unreachable (timed out after 15 seconds, no retries).")
else:
    print("=== Socrata 311 API Inspection (Light Call: 2 Requests) ===")
    api_rows = []
    for ds_id, info in api_results.items():
        row = {
            "Dataset ID": ds_id,
            "Status": info["status_code"],
            "Sample Fields": info["sample_field_count"],
            "created_date": info["fields_present"]["created_date"],
            "complaint_type": info["fields_present"]["complaint_type"],
            "descriptor": info["fields_present"]["descriptor"],
            "borough": info["fields_present"]["borough"],
        }
        api_rows.append(row)
    api_df = pd.DataFrame(api_rows)
    print(api_df.to_string(index=False))

=== Socrata 311 API Inspection (Light Call: 2 Requests) ===
Dataset ID  Status  Sample Fields  created_date  complaint_type  descriptor  borough
 76ig-c548     503              0         False           False       False    False
 erm2-nwe9     200             23          True            True        True     True


In [7]:
checker = Checker()
checker.check("header has 44 columns", raw_inspection["header_count"], 44)
checker.check("date-parse failures on the sample = 0", raw_inspection["date_parse_failures"], 0)
checker.summary()

[PASS] header has 44 columns: actual=44, expected=44
[PASS] date-parse failures on the sample = 0: actual=0, expected=0
CHECKS: 2 passed, 0 failed


In [8]:
findings = f"""### Findings
- **Environment**: Python `{env_info['python']}`, pandas `{env_info['pandas']}`, numpy `{env_info['numpy']}`, scikit-learn `{env_info['scikit-learn']}`, PyTorch `{env_info['torch']}` on branch `{env_info['git_branch']}` with {env_info['free_disk_gb']} GB free disk.
- **Raw CSV Header**: Contains exactly {raw_inspection['header_count']} columns, with all 12 key concept mappings present (including 'Problem (formerly Complaint Type)' and 'Problem Detail (formerly Descriptor)').
- **Sample Parsing**: Exactly {raw_inspection['date_parse_failures']} parse failures observed on the 5-row sample using format `%m/%d/%Y %I:%M:%S %p`.
- **File Completeness**: Binary inspection revealed the raw 2020 CSV ends with `{raw_inspection['last_3_lines'][-1].strip()}`, confirming: {raw_inspection['verdict']}.
- **Processed Coverage**: Aggregated cube covers {cube_info['first_month']} through {cube_info['last_month']}, with partial month boundaries at {cube_info['partial_months'][0]} and {cube_info['partial_months'][1]}.
"""
md(findings)
checker.summary()

### Findings
- **Environment**: Python `3.11.18`, pandas `3.1.1`, numpy `2.4.6`, scikit-learn `1.9.1`, PyTorch `2.15.0+cpu` on branch `data-ml` with 25.95 GB free disk.
- **Raw CSV Header**: Contains exactly 44 columns, with all 12 key concept mappings present (including 'Problem (formerly Complaint Type)' and 'Problem Detail (formerly Descriptor)').
- **Sample Parsing**: Exactly 0 parse failures observed on the 5-row sample using format `%m/%d/%Y %I:%M:%S %p`.
- **File Completeness**: Binary inspection revealed the raw 2020 CSV ends with `}`, confirming: INCOMPLETE / SERVER ERROR (ends with Socrata 500 error JSON payload).
- **Processed Coverage**: Aggregated cube covers 2025-10 through 2026-11, with partial month boundaries at 2025-10 and 2026-11.


CHECKS: 2 passed, 0 failed


### Limitations

- The raw CSV download from Socrata was interrupted before full completion, terminating with an HTTP 500 error payload. Downstream workflows must rely on verified pre-extracted subsets and precomputed cubes rather than re-scanning the truncated raw export.
- Remote Socrata API requests can experience high network latency or timeouts under heavy server load, necessitating local caching and offline fallback flags.
- Edge calendar months (October 2025 and November 2026) are partial observation periods and must be excluded from baseline monthly median calculations.